# NLP Text Classification
## Banking Use Case: Classifying customer emails and detecting complaint sentiment

**What this notebook covers:**

Classical NLP pipeline — the foundation before LLMs.
Understanding this pipeline helps you explain *why* LLMs are powerful
and what problems they solve that classical methods struggled with.

**Pipeline steps:**
1. Text preprocessing — cleaning, tokenisation, normalisation
2. Feature extraction — TF-IDF vectorisation
3. Classification — logistic regression and random forest
4. Evaluation — accuracy, confusion matrix, classification report
5. Comparison — classical NLP vs Hugging Face transformer

**Banking relevance:**
- Route incoming customer emails to the right department automatically
- Flag urgent or high-risk complaints for immediate attention
- Classify transaction descriptions for categorisation
- Screen documents for compliance-relevant language

**To adapt this notebook:**
- Replace the sample texts with your own dataset
- Add or remove categories to match your classification task
- Swap the classifier for any scikit-learn compatible model


In [ ]:
%pip install scikit-learn pandas numpy matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
print("Libraries loaded")

## Step 1 — Create a labelled dataset

In a real project you would load labelled data from a CSV or database.
Here we create a synthetic but realistic dataset of banking customer messages.
This gives you full control for learning and demonstration purposes.

In [ ]:
# Labelled customer messages
# Each entry: (text, category)
# Categories: FRAUD, LOAN, ACCOUNT, PAYMENT, GENERAL

data = [
    # FRAUD
    ("I noticed a charge I did not make on my account yesterday.", "FRAUD"),
    ("Someone used my card without my permission.", "FRAUD"),
    ("There is a suspicious transaction of EUR 200 I do not recognise.", "FRAUD"),
    ("I think my card details have been stolen and used online.", "FRAUD"),
    ("An unauthorised payment was made from my account last night.", "FRAUD"),
    ("I received a text about a payment I never made.", "FRAUD"),
    ("My card was used at a shop in another city when I was at home.", "FRAUD"),
    ("There are two transactions on my statement I cannot explain.", "FRAUD"),
    ("I lost my wallet and someone may have used my bank card.", "FRAUD"),
    ("A payment was made to an account I do not know.", "FRAUD"),
    # LOAN
    ("I would like to apply for a personal loan.", "LOAN"),
    ("What is the interest rate for a EUR 10,000 loan over 36 months?", "LOAN"),
    ("Can I get a loan to renovate my apartment?", "LOAN"),
    ("I want to refinance my existing loan at a lower rate.", "LOAN"),
    ("How long does it take to process a loan application?", "LOAN"),
    ("What documents do I need for a loan?", "LOAN"),
    ("I would like to increase my credit limit.", "LOAN"),
    ("Can I repay my loan early without a penalty?", "LOAN"),
    ("My loan application was rejected and I want to know why.", "LOAN"),
    ("I need a loan urgently to cover unexpected medical expenses.", "LOAN"),
    # ACCOUNT
    ("I want to change my address on my account.", "ACCOUNT"),
    ("How do I add a joint account holder?", "ACCOUNT"),
    ("I want to close my account and move to another bank.", "ACCOUNT"),
    ("I forgot my PIN and cannot access my account.", "ACCOUNT"),
    ("Please update my email address on file.", "ACCOUNT"),
    ("I need a copy of my bank statement for the last six months.", "ACCOUNT"),
    ("How do I set up online banking?", "ACCOUNT"),
    ("I want to open a savings account.", "ACCOUNT"),
    ("My account has been blocked and I do not know why.", "ACCOUNT"),
    ("I need to update my phone number for two-factor authentication.", "ACCOUNT"),
    # PAYMENT
    ("My salary transfer has not arrived yet.", "PAYMENT"),
    ("I made a payment to the wrong account by mistake.", "PAYMENT"),
    ("Can you confirm that my payment has been processed?", "PAYMENT"),
    ("I set up a standing order but it did not execute.", "PAYMENT"),
    ("How long does an international transfer take?", "PAYMENT"),
    ("I want to cancel a direct debit.", "PAYMENT"),
    ("A payment was returned but I have not received a refund.", "PAYMENT"),
    ("I need to make an urgent same-day transfer.", "PAYMENT"),
    ("My regular payment to my landlord failed this month.", "PAYMENT"),
    ("I transferred money but the recipient says they did not receive it.", "PAYMENT"),
    # GENERAL
    ("What are your branch opening hours?", "GENERAL"),
    ("Do you have ATMs near the city centre?", "GENERAL"),
    ("What exchange rate do you offer for US dollars?", "GENERAL"),
    ("Do you offer student accounts?", "GENERAL"),
    ("How do I contact customer support by phone?", "GENERAL"),
    ("Is there a fee for using my card abroad?", "GENERAL"),
    ("What savings products do you currently offer?", "GENERAL"),
    ("Do you have a mobile banking app?", "GENERAL"),
    ("Can I get a bank reference letter for a visa application?", "GENERAL"),
    ("What is your IBAN for receiving international payments?", "GENERAL"),
]

df = pd.DataFrame(data, columns=["text", "category"])
print(f"Dataset: {len(df)} examples across {df['category'].nunique()} categories")
print(df["category"].value_counts())

## Step 2 — Text preprocessing

Before vectorising, we clean the text.
For TF-IDF this matters less than for older methods,
but understanding these steps is essential for explaining
what vectorisers do under the hood.

In [ ]:
import re
import string

def preprocess(text: str) -> str:
    """
    Clean and normalise a text string.
    Steps:
    1. Lowercase — 'Loan' and 'loan' are the same word
    2. Remove punctuation — TF-IDF does not need it
    3. Remove extra whitespace

    Note: for production NLP you would also consider:
    - Stopword removal (e.g. removing 'the', 'a', 'I')
    - Lemmatisation (reducing 'running' to 'run')
    - Handling currency symbols and numbers specially
    TF-IDF's sublinear_tf and max_df parameters partially compensate
    for not doing these steps explicitly.
    """
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply preprocessing
df["text_clean"] = df["text"].apply(preprocess)

# Show the effect
print("Before:", df["text"].iloc[0])
print("After: ", df["text_clean"].iloc[0])

## Step 3 — TF-IDF Vectorisation

**TF-IDF (Term Frequency-Inverse Document Frequency)**
converts text into a numerical matrix where:
- Each row = one document
- Each column = one word (or n-gram)
- Each value = how important that word is to that document

Words that appear everywhere (like 'the') get low scores.
Words that are distinctive to a category get high scores.
This is how classical NLP finds signal in text.

In [ ]:
# Fit TF-IDF on training data only
vectoriser = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=500,
    sublinear_tf=True
)

X_train_tfidf = vectoriser.fit_transform(X_train)
X_test_tfidf = vectoriser.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"({X_train_tfidf.shape[0]} documents × {X_train_tfidf.shape[1]} features)")

# Show the most distinctive terms per category
print("\nTop TF-IDF terms by category:")
feature_names = vectoriser.get_feature_names_out()

for category in sorted(df["category"].unique()):
    # Fix: use .values to get numpy array, then boolean index works correctly
    category_mask = (y_train.values == category)

    if category_mask.sum() == 0:
        continue

    # Fix: use .toarray() instead of .A1 for newer scipy versions
    category_matrix = X_train_tfidf[category_mask]
    category_tfidf = category_matrix.toarray().mean(axis=0)

    top_indices = category_tfidf.argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    print(f"  {category:10}: {', '.join(top_terms)}")

## Step 4 — Train and evaluate classifiers

In [ ]:
# We use sklearn Pipeline to bundle vectoriser + classifier
# This prevents data leakage and simplifies deployment

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
}

results = {}

for name, clf in classifiers.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=500, sublinear_tf=True)),
        ("clf", clf)
    ])
    # Cross-validation on training data
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    # Final fit
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    test_acc = (y_pred == y_test).mean()
    results[name] = {"pipeline": pipe, "y_pred": y_pred, "cv_mean": cv_scores.mean()}
    print(f"{name:25} | CV Accuracy: {cv_scores.mean():.2%} ± {cv_scores.std():.2%} | Test: {test_acc:.2%}")

# Detailed report for the best model
best_name = max(results, key=lambda x: results[x]["cv_mean"])
best_pred = results[best_name]["y_pred"]
print(f"\n=== {best_name} — Classification Report ===")
print(classification_report(y_test, best_pred))

In [ ]:
# Confusion matrix
categories = sorted(df["category"].unique())
cm = confusion_matrix(y_test, best_pred, labels=categories)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=categories, yticklabels=categories
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## Step 5 — Test on new inputs

Try the model on sentences it has never seen.
This is also a good way to find failure modes.

In [ ]:
best_pipeline = results[best_name]["pipeline"]

new_messages = [
    "Someone made a purchase of EUR 300 on my card without my knowledge.",
    "I would like to know your current mortgage rates.",
    "My direct debit for the gym did not go through this month.",
    "I want to update my next of kin details on my account.",
    "Where is the nearest branch to the first district?",
    # Try an ambiguous one
    "I need money urgently.",
]

print(f"Predictions using: {best_name}")
print("-" * 70)
for msg in new_messages:
    prediction = best_pipeline.predict([msg])[0]
    probabilities = best_pipeline.predict_proba([msg])[0]
    confidence = max(probabilities)
    print(f"Message:    {msg}")
    print(f"Prediction: {prediction} (confidence: {confidence:.1%})")
    print()

## Step 6 — Classical NLP vs Transformer (Hugging Face)

This comparison shows why transformers replaced TF-IDF for most NLP tasks —
and also when classical methods are still the right choice.

In [ ]:
# Hugging Face zero-shot classifier — no training data needed
# Install transformers if not already installed
# %pip install transformers torch --quiet

try:
    from transformers import pipeline as hf_pipeline

    classifier_hf = hf_pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    labels = ["fraud", "loan", "account management", "payment", "general inquiry"]
    test_message = "Someone used my card without my permission at a shop abroad."

    # Classical model
    classical_pred = best_pipeline.predict([test_message])[0]
    classical_conf = max(best_pipeline.predict_proba([test_message])[0])

    # Transformer model
    hf_result = classifier_hf(test_message, labels)
    hf_pred = hf_result["labels"][0]
    hf_conf = hf_result["scores"][0]

    print(f"Message: {test_message}")
    print(f"\nClassical TF-IDF + LR: {classical_pred} ({classical_conf:.1%} confidence)")
    print(f"Hugging Face zero-shot: {hf_pred} ({hf_conf:.1%} confidence)")

except ImportError:
    print("transformers not installed — run: pip install transformers torch")

## Summary

| | Classical NLP (TF-IDF + LR) | Transformer (Hugging Face) |
|---|---|---|
| **Training data needed** | Yes — labelled examples | No — zero-shot possible |
| **Speed** | Very fast | Slower |
| **Interpretability** | High — inspect feature weights | Low — black box |
| **Context understanding** | Limited — bag of words | High — full sentence context |
| **Setup complexity** | Low | Medium |
| **Cost** | Free | API cost or compute |
| **Best for** | High-volume, stable categories | Novel categories, nuanced text |

**When to use classical NLP:**
- You have labelled data and stable categories
- Speed and cost matter (high-volume processing)
- Interpretability is required (regulatory context)
- You need to explain feature weights to stakeholders

**When to use transformers:**
- Categories are new or undefined
- Text is nuanced or context-dependent
- You do not have labelled training data
- Accuracy matters more than speed
